In [0]:
import importlib
import sys

# 1
if "../src" not in sys.path:
    sys.path.append("../src")

import bronze.nasa_firms_volume_ingestor as firms_mod

importlib.reload(firms_mod)

OUTPUT_JSON = "/Volumes/workspace/default/bronze_nasa_firms/data-2015-2026.json"

# 2
new_rows = firms_mod.ingest_firms_incremental(
    date_from="2026-11-20",
    date_to="2026-11-22",
    output_file=OUTPUT_JSON,
    source="VIIRS_NOAA20_NRT",
    splice_label="incremental_nrt",
    step_days=1,
    sleep_seconds=0.4,
)
print(f"Filas añadidas en este incremental: {new_rows}")

# 3
df_bronze = spark.read.json(OUTPUT_JSON)
print("Total filas en Bronze:", df_bronze.count())
df_bronze.selectExpr("min(acq_date) as fecha_min", "max(acq_date) as fecha_max").show()
df_bronze.groupBy("firms_source", "splice_segment").count().orderBy("firms_source", "splice_segment").show()

# --- Instrucción para regenerar los 11 años desde cero ---
# all_data = firms_mod.ingest_firms_spliced_2015_2026(
#     output_file=OUTPUT_JSON,
#     step_days=5,
#     sleep_seconds=0.5,
#     reset_file=True,
# )
# print(all_data)

In [0]:
import importlib
import sys

SRC_PATH = "/Workspace/Shared/extremadura-wildfire-intelligence/src"
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

sys.modules.pop("silver.firms_cleaner", None)
import silver.firms_cleaner as firms_silver
importlib.reload(firms_silver)

df_bronze = spark.read.json("/Volumes/workspace/default/bronze_nasa_firms/data-2015-2026.json")
df_step1 = firms_silver.cast_and_parse_timestamps(df_bronze)
df_step2 = firms_silver.enrich_confidence(df_step1)
df_step3 = firms_silver.filter_valid_detections(df_step2)
df_step4 = firms_silver.deduplicate_detections(df_step3)

print("Filas antes de filtrar (df_step2):", df_step2.count())
print("Filas después de filtrar (df_step3):", df_step3.count())
print("Filas después de deduplicar (df_step4):", df_step4.count())